# LLM Inference Lab: Kaggle runner

Thin wrapper around the repo scripts. All the logic lives in `scripts/`; this notebook only calls it.

**Before running:** Settings → Accelerator **GPU T4 x1**, Internet **on**.
After the first quantization, save `/kaggle/working/models` as a private dataset named `llm-lab-models`
and add it as an input, so later sessions skip stage A.

Run cell 1, then only the stage you need. Every stage is safe to re-run: finished work is skipped.

In [ ]:
# 1. Setup (every session: pip installs don't survive a restart)
import os, subprocess
REPO = "/kaggle/working/llm-inference-lab"
if not os.path.exists(REPO):
    !git clone -q https://github.com/tomaskier/llm-inference-lab.git {REPO}
else:
    # fetch + reset instead of pull, so it also works if history was rewritten
    !git -C {REPO} fetch -q && git -C {REPO} reset -q --hard origin/main
os.chdir(REPO)

# Quantized models: from the input dataset if attached, else from this session's output.
DATASET = "/kaggle/input/llm-lab-models"
os.environ["MODELS_DIR"] = DATASET if os.path.exists(DATASET) else "/kaggle/working/models"
print("MODELS_DIR =", os.environ["MODELS_DIR"])

!bash scripts/kaggle_setup.sh

## A. Quantize (first session only, ~20 min)

In [ ]:
os.environ["MODELS_DIR"] = "/kaggle/working/models"
!make quantize VARIANT=w8
!make quantize VARIANT=w4
!du -sh /kaggle/working/models/*

## B. Size, memory, perplexity (~10 min per variant)

In [ ]:
for v in ["base", "w8", "w4"]:
    if os.path.exists(f"results/quantization/{v}.json"):
        print("skip", v); continue
    !make serve VARIANT={v} CACHE=off
    try:
        !make smoke
        !make measure VARIANT={v}
    finally:
        !make stop

## C. Accuracy with lm-eval (~1 h per variant)

In [ ]:
for v in ["base", "w8", "w4"]:
    if os.path.exists(f"results/eval/{v}.json"):
        print("skip", v); continue
    !make serve VARIANT={v} CACHE=off
    try:
        !make eval VARIANT={v}
    finally:
        !make stop

## D. Benchmark matrix (~6 h in total, resumable)

Split it across sessions if needed, e.g. `--variants base` in one session and the rest in the next.

In [ ]:
!python scripts/run_matrix.py --variants base w8 w4

## E. Observability session (optional)

Starts vLLM, a small proxy that exposes **only** `/metrics` on port 9101, and a cloudflared tunnel to
that proxy. The OpenAI API itself stays on localhost. Put the printed host in
`observability/targets/vllm.json` on your laptop, run the load test cell and watch Grafana.

In [ ]:
import re, time
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared

!make serve VARIANT=base CACHE=on
proxy = subprocess.Popen(["python", "scripts/metrics_proxy.py", "--port", "9101"])
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:9101"],
                          stdout=open("build/tunnel.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://([\w-]+\.trycloudflare\.com)", open("build/tunnel.log").read())
    if m: break
print("Tunnel host:", m.group(1) if m else "not found, see build/tunnel.log")
!python scripts/check_metrics.py

In [ ]:
# Load test that should trip QueueBuilding, KVCacheNearFull and HighTTFT: long prompts, 64 users, 5 minutes.
!.venvs/bench/bin/guidellm run --backend kind=openai_http,target=http://localhost:8000,request_format=/v1/completions   --profile '{"kind": "concurrent", "streams": [64]}' --constraint kind=max_duration,seconds=300   --data '{"kind": "synthetic_text", "prompt_tokens": 4096, "output_tokens": 512}' --disable-console-interactive   --output kind=json,path=build/loadtest.json

In [ ]:
# Close the tunnel, the proxy and the server when done.
tunnel.terminate(); proxy.terminate()
!make stop

## F. Pack results for download

In [ ]:
!python scripts/check_regression.py --results results --partial || true
!cd /kaggle/working && zip -qr results.zip llm-inference-lab/results llm-inference-lab/build/vllm.log
print("Download /kaggle/working/results.zip from the Output panel, unzip it into the repo and commit results/.")